In [1]:
import json
import concrete_ml_extensions
import concrete
import json
import concrete_ml_extensions as fhext
import json
from pathlib import Path 
from concrete import fhe 
import pickle as pkl 
import subprocess

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from concrete.ml.sklearn import DecisionTreeClassifier as ConcreteDecisionTreeClassifier
from sklearn.model_selection import train_test_split
from concrete.ml.common.utils import CiphertextFormat

assert concrete.ml.__version__ == "1.8.0"
assert concrete_ml_extensions.__version__ == "0.1.6"

x, y = make_classification(n_samples=100, class_sep=2, n_features=30, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(y_test)

model = ConcreteDecisionTreeClassifier(n_bits=8)
model.fit(X_train, y_train)

model.compile(X_train[:10, :], ciphertext_format=CiphertextFormat.TFHE_RS)

crypto_params = json.loads(fhext.get_crypto_params_radix())

print(crypto_params)

sk, tfhers_pk, pk, lwe_sk = model.keygen_tfhers()


with open("evalkeys_concrete.bin", "wb") as f:
    f.write(tfhers_pk.serialize())

with open("evalkeys_tfhers.bin", "wb") as f:
    f.write(pk)

path_circuit_server = Path("server").joinpath("server.zip")
model.fhe_circuit.server.save(path_circuit_server, via_mlir=True)

[0 0 0 0 1 0 1 0 1 1 0 0 1 0 0 1 1 1 0 0]
----> {'lwe_dimension': 834, 'glwe_dimension': 1, 'polynomial_size': 2048, 'lwe_noise_distribution': {'Gaussian': {'std': 3.5539902359442825e-06, 'mean': 0.0}}, 'glwe_noise_distribution': {'Gaussian': {'std': 2.845267479601915e-15, 'mean': 0.0}}, 'pbs_base_log': 23, 'pbs_level': 1, 'ks_base_log': 3, 'ks_level': 5, 'message_modulus': 4, 'carry_modulus': 4, 'max_noise_level': 5, 'log2_p_fail': -64.074, 'ciphertext_modulus': {'modulus': 0, 'scalar_bits': 64}, 'encryption_key_choice': 'Big', 'modulus_switch_noise_reduction_params': None}
{'lwe_dimension': 834, 'glwe_dimension': 1, 'polynomial_size': 2048, 'lwe_noise_distribution': {'Gaussian': {'std': 3.5539902359442825e-06, 'mean': 0.0}}, 'glwe_noise_distribution': {'Gaussian': {'std': 2.845267479601915e-15, 'mean': 0.0}}, 'pbs_base_log': 23, 'pbs_level': 1, 'ks_base_log': 3, 'ks_level': 5, 'message_modulus': 4, 'carry_modulus': 4, 'max_noise_level': 5, 'log2_p_fail': -64.074, 'ciphertext_modulus'

In [2]:
server = fhe.Server.load("server/server.zip")

with open("evalkeys_concrete.bin", "rb") as f:
    b = f.read()

evalKeys = fhe.EvaluationKeys.deserialize(b)

In [3]:
# tree decision is 1, TFHE-rs code returns the random token
# tree decision is 0, TFHE-rs code returns 0s
for DATA_INDEX in range(10):
    print("\nExample ", DATA_INDEX)
    q_input = model.quantize_input(X_test[[DATA_INDEX]])
    print("Inference inputs:", q_input, " | Expected output", y_test[DATA_INDEX])
    q_input_enc = model.encrypt_tfhers(q_input, tfhers_sk=sk)

    with open("input.bin", "wb") as f:
        f.write(q_input_enc[0].serialize())

    # Server circuit run - like in deployment
    # 
    with open("input.bin", "rb") as f:
        b = f.read()
    value = fhe.Value.deserialize(b)

    output = server.run(value, evaluation_keys=evalKeys)
    
    print("Circuit output: ", model.decrypt_tfhers(output, sk).reshape((-1,)))

    with open("prediction_non_preprocessed.bin", "wb") as f:
        f.write(model._tfhers_bridge.export_value(output, 0))

    # ***************************************
    # NOW RUN cargo run --release
    # **************************************

    result = subprocess.run(["cargo", "run", "--release"], capture_output=True, text=True)

    tfhers_postproc_bytes = open("tfhers_sign_result.bin", "rb").read()
    results = fhext.decrypt_serialized_i8_radix_2d(tfhers_postproc_bytes, 16, sk)
    print("Authentication token: ", results.reshape((-1,)))


Example  0
Inference inputs: [[177 128 122  23  88  65 139  15 166 131  40  64  67 255 128  24  40 117
  142 226 104 138 114 152  40 128 156 205 215  64]]  | Expected output 0
Circuit output:  [255   0]
Authentication token:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Example  1
Inference inputs: [[177 141 156 112 153 115 134 164 148 150 140 105 119 167 163 131 111 118
  125 162  99 158 106 233  94 213 137 142 161  88]]  | Expected output 0
Circuit output:  [255   0]
Authentication token:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Example  2
Inference inputs: [[167  59  47 151 148 141 165  99 138 132 110 133 228  95 111 130 211  67
  108 156  57 186 177 149  64 163 144  41  69 185]]  | Expected output 0
Circuit output:  [255   0]
Authentication token:  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

Example  3
Inference inputs: [[100 161 144 165 179 123  50 132  82 170 131 180 209 127  99 142 182  94
  109 189 150  95  77  69  33 141 110 148  75 172]]  | Expected output 0
Circuit output:  [255   0]
Authenticatio